In [ ]:
%pip install --quiet pyeyesweb[datasets] matplotlib pandas
%matplotlib inline
# helpers.py (the plotting functions of the tutorials) is fetched once, next to this notebook
from pathlib import Path
if not Path('helpers.py').exists():
    import urllib.request
    urllib.request.urlretrieve('https://raw.githubusercontent.com/InfoMusCP/PyEyesWeb/main/examples/helpers.py', 'helpers.py')


# Synchronization of continuous signals

Two hands that rise and fall together, a group that breathes as one: synchronization is how far several movements are
coordinated in time. This first of two tutorials compares **continuous signals**: it measures phase locking, correlation,
partial correlation, nonlinear interdependence and recurrence, on a performer who moves both arms together (synchronised) or
not (unsynchronised).

:Goals: measure the synchronization of two signals and of a group with seven features; read a recurrence plot; know what each
        measure compares and what its output looks like in time.
:Prerequisites: `sphx_glr_tutorials_05_analysis_primitives_plot_08_primitives.py` (windows, statuses and the
                analysis primitives).
:Data: trials 5 (breathing arms, synchronised) and 6 (unsynchronised), seconds 5 to 21 of the example session. The tutorial
       after this one compares *events* (`sphx_glr_tutorials_05_analysis_primitives_plot_09b_synchronization_events.py`).
:Time: about 25 minutes.


## Data used in this tutorial
In the breathing-arms task the performer raises and lowers both arms in a slow cycle. In trial 5 the two arms move together,
in trial 6 they do not (see ``examples/data/README.md``). The two "participants" of every measure below are the left and the
right hand. Each is reduced to one number per sample, **its height** in metres (the vertical axis, ``y`` in these recordings),
because the synchronization measures compare one signal per participant (the theory is in
:doc:`/theory/synchronization`). Time is in seconds; the window parameters of the features are in samples (100 per second).



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from helpers import shade_status
from pyeyesweb.datasets import load_qualisys_trial

trials = {"trial 5 (synchronised)": 5, "trial 6 (unsynchronised)": 6}
heights = {}
for name, number in trials.items():
    trial = load_qualisys_trial(number, start=5, stop=21).to_units("m")
    heights[name] = np.column_stack([trial.joint("Left_Hand")[:, 1], trial.joint("Right_Hand")[:, 1]])
    stamps = trial.timestamps
sampling_rate = trial.sampling_rate
print("samples per trial:", len(stamps), "| heights of the two hands:", heights["trial 5 (synchronised)"].shape)

fig, axes = plt.subplots(2, 1, figsize=(8, 4), sharex=True, sharey=True)
for ax, (name, signal) in zip(axes, heights.items()):
    ax.plot(stamps, signal[:, 0], label="left hand")
    ax.plot(stamps, signal[:, 1], label="right hand")
    ax.set_ylabel("height (m)")
    ax.set_title(name, fontsize=9, loc="left")
axes[0].legend(fontsize=7, ncol=2)
axes[-1].set_xlabel("time (s)")
fig.tight_layout()

**Takeaway.** In the synchronised trial the two hands follow the same curve; in the other they drift apart.



## Phase locking: do the rhythms stay in step?
:class:`~pyeyesweb.analysis_primitives.PhaseLockingValue` (PLV) turns each signal into a phase, which says where the
signal is in its cycle, and measures how constant the *difference* of the two phases stays over a window: 1 when the
difference does not change (the signals are in step, whatever their amplitudes or a constant lag), near 0 when it wanders.
The phase is only meaningful for a signal dominated by one rhythm, so the feature can band-pass the signals first: give
``low_frequency`` and ``high_frequency`` (in Hz) and the ``sampling_rate``. Without them nothing is filtered.

The call takes one sample of each signal (``process(first, second)``) or two series (``run(first, second)``). We use a
window of 3 s that moves by 0.5 s, with and without a band of 0.5 to 3 Hz.



In [ ]:
from pyeyesweb.analysis_primitives import PhaseLockingValue

curves = {}
for name, signal in heights.items():
    for label, band in (("no filter", {}), ("band 0.5-3 Hz", dict(low_frequency=0.5, high_frequency=3.0, sampling_rate=sampling_rate))):
        curves[name, label] = PhaseLockingValue(window=300, hop=50, **band).run(signal[:, 0], signal[:, 1], timestamps=stamps)
        recorder = curves[name, label]
        print(f"{name:<26} {label:<14} mean PLV {np.mean(recorder.phase_locking_value[recorder.ok]):.2f}")

fig, axes = plt.subplots(1, 2, figsize=(8, 3), sharey=True)
for ax, label in zip(axes, ("no filter", "band 0.5-3 Hz")):
    for (name, key), recorder in curves.items():
        if key == label:
            ax.plot(recorder.timestamps[recorder.ok], recorder.phase_locking_value[recorder.ok], label=name)
    shade_status(ax, stamps, curves["trial 5 (synchronised)", label].status, codes=(1,))
    ax.set_title(label, fontsize=9, loc="left")
    ax.set_xlabel("time (s)")
    ax.set_ylim(0, 1.05)
axes[0].set_ylabel("PLV")
axes[0].legend(fontsize=7, loc="lower right")
fig.tight_layout()

The synchronised trial stays higher. The yellow band is the warm-up, while the first window fills. Filtering lowers the
values, because the phase then follows only the movement in the band; compare values computed with the same settings.

**Takeaway.** PLV is 1 when two rhythms stay in step; give the frequency band of the movement for a meaningful phase.



## Nonlinear interdependence: does one signal determine the other?
Correlation sees only straight-line relations. :class:`~pyeyesweb.analysis_primitives.NonlinearInterdependence` asks
whether the *state* of one signal, a short stretch of ``embedding_dimension`` samples, determines the state of the other: the
``neighbours`` nearest states of one signal are looked up in the other. It returns the two directions
(``first_to_second_interdependence`` and the reverse; they may differ, and the difference is read as the direction of
the coupling) and a ``threshold`` computed from the parameters. Identical signals give 1 and independent signals near 0.



In [ ]:
from pyeyesweb.analysis_primitives import NonlinearInterdependence

fig, ax = plt.subplots(figsize=(8, 3))
for name, signal in heights.items():
    recorder = NonlinearInterdependence(window=300, hop=100, embedding_dimension=3, neighbours=5).run(signal[:, 0], signal[:, 1], timestamps=stamps)
    ax.plot(recorder.timestamps[recorder.ok], recorder.first_to_second_interdependence[recorder.ok], "o-", ms=3, label=f"{name}, left to right")
    ax.plot(recorder.timestamps[recorder.ok], recorder.second_to_first_interdependence[recorder.ok], "s--", ms=3, label=f"{name}, right to left")
    print(f"{name:<26} mean interdependence {np.mean(recorder.first_to_second_interdependence[recorder.ok]):.2f}, threshold {recorder.threshold[recorder.ok][0]:.2f}")
ax.set_xlabel("time (s)")
ax.set_ylabel("interdependence")
ax.legend(fontsize=7, ncol=2)
fig.tight_layout()

**Takeaway.** Interdependence is higher when the movement of one hand determines that of the other, linearly or not.



## A group: correlation and direct synchronization
For more than two signals the question is who moves with whom.
:class:`~pyeyesweb.analysis_primitives.CorrelationsMatrix` computes, over a window, the Pearson correlation of every pair
of ``number_of_inputs`` signals, with negative values set to 0 (anti-correlation is not synchronization). Two signals can
correlate only because both follow a third;
:class:`~pyeyesweb.analysis_primitives.PartialRecurrenceSynchronization` takes that matrix and removes what the other
signals explain, leaving the *direct* relation of each pair (it has no window of its own: one matrix in, one matrix out).

The four signals are the heights of both hands, the left knee and the head.



In [ ]:
from pyeyesweb.analysis_primitives import CorrelationsMatrix, PartialRecurrenceSynchronization

parts = ["Left_Hand", "Right_Hand", "Left_Knee", "Head"]
matrices = {}
for name, number in trials.items():
    trial = load_qualisys_trial(number, start=5, stop=21).to_units("m")
    group = np.column_stack([trial.joint(part)[:, 1] for part in parts])
    correlations = CorrelationsMatrix(window=1600, number_of_inputs=4).run(group)[-1].correlations_matrix
    direct = PartialRecurrenceSynchronization().process(correlations).synchronization_matrix
    matrices[name] = (correlations, direct)

fig, axes = plt.subplots(2, 2, figsize=(7, 6.4))
for row, (name, (correlations, direct)) in enumerate(matrices.items()):
    for ax, matrix, title in zip(axes[row], (correlations, direct), ("correlation", "direct (partial)")):
        image = ax.imshow(matrix, vmin=0, vmax=1, cmap="viridis")
        ax.set_xticks(range(4), [part.replace("_", "\n") for part in parts], fontsize=7)
        ax.set_yticks(range(4), [part.replace("_", " ") for part in parts] if ax in axes[:, 0] else [], fontsize=7)
        ax.set_title(f"{name}: {title}", fontsize=8)
        ax.grid(False)
fig.colorbar(image, ax=axes, shrink=0.8, label="value (dimensionless)")

The two hands are bound together in the synchronised trial and not in the other. The second column keeps only what a pair
shares directly: for the hands and the knee, part of the correlation seen in the first column is shared through the other
signals.

**Takeaway.** The correlation matrix says who moves with whom; the partial matrix says who is *directly* linked.



## Recurrence: when does the movement come back?
A movement that repeats returns to the same states. :class:`~pyeyesweb.analysis_primitives.GeneralisedAutoCorrelation`
measures, for each lag, the fraction of samples that return to within ``epsilon`` of where they were that many samples
earlier. It peaks at the lags at which the movement repeats. A repetition that is quicker than the slow arm cycle shows it
better, so here we use the horizontal position of the right hand in the clapping task, trial 7 (synchronised) and trial 8
(unsynchronised), over 10 s, with lags from 0.01 s to 3 s.



In [ ]:
from pyeyesweb.analysis_primitives import GeneralisedAutoCorrelation

clapping = {name: load_qualisys_trial(number, start=5, stop=21).to_units("m").joint("Right_Hand")[:1000, 0:1]
            for name, number in {"trial 7 (synchronised)": 7, "trial 8 (unsynchronised)": 8}.items()}
fig, ax = plt.subplots(figsize=(8, 3))
for name, position in clapping.items():
    rates = GeneralisedAutoCorrelation(window=1000, epsilon=0.05, tau_min=1, tau_spread=300).run(position)[-1]
    ax.plot(rates.taus / sampling_rate, rates.recurrence_rate, label=name)
ax.set_xlabel("lag (s)")
ax.set_ylabel("recurrence rate")
ax.legend(fontsize=7)
fig.tight_layout()

The regular clapping returns to the same positions at regular lags; the unsynchronised trial has no clear peaks. With
``output_type="scalar"`` and a ``tau`` the feature returns the rate at one lag, which is convenient for a live loop. The lags
(``taus``) are in samples; divide by the sampling rate for seconds.

The recurrence of a window can also be drawn. :class:`~pyeyesweb.analysis_primitives.RecurrencePlot` is the binary matrix
whose entry (i, j) is 1 when the state at sample i is within ``epsilon`` of the state at sample j: diagonal lines show that
the movement repeats. :class:`~pyeyesweb.analysis_primitives.CrossRecurrencePlot` compares two signals instead: a strong main diagonal
means that the two follow the same trajectory at the same time, and a line displaced from the diagonal means a delay.
Windows of 5 s (500 samples).



In [ ]:
from pyeyesweb.analysis_primitives import CrossRecurrencePlot, RecurrencePlot

fig, axes = plt.subplots(2, 3, figsize=(8, 5.4))
density = {}
for row, (name, signal) in zip(axes, heights.items()):
    own = RecurrencePlot(window=500, epsilon=0.05).run(signal[:500, 0:1])[-1].recurrence_matrix
    cross = CrossRecurrencePlot(window=500, epsilon=0.05).run(signal[:500, 0:1], signal[:500, 1:2])[-1].cross_recurrence_matrix
    row[0].imshow(own, origin="lower", cmap="Greys", extent=[0, 5, 0, 5])
    row[1].imshow(cross, origin="lower", cmap="Greys", extent=[0, 5, 0, 5])
    row[2].plot(np.arange(500) / sampling_rate, np.diag(cross), lw=0.6)
    row[0].set_title(f"{name}\nrecurrence, left hand", fontsize=8)
    row[1].set_title("cross-recurrence, left x right", fontsize=8)
    row[2].set_title("main diagonal of the cross plot", fontsize=8)
    for ax in row[:2]:
        ax.grid(False)
        ax.set_xlabel("time j (s)")
    row[0].set_ylabel("time i (s)")
    row[2].set_xlabel("time (s)")
    density[name] = float(np.diag(cross).mean())
fig.tight_layout()
print("share of the main diagonal of the cross plot that recurs:", {name: round(value, 2) for name, value in density.items()})

**Takeaway.** Recurrence shows when a movement returns to a posture; the cross-recurrence diagonal is full when two movements
are the same at the same time.



## Side by side
Each measure for the left and right hands: PLV and correlation over the 16 s, interdependence of the last 3 s window,
cross-recurrence of the first 5 s. Values are comparable only within a row.



In [ ]:
rows = {}
for name, signal in heights.items():
    left, right = signal[:, 0], signal[:, 1]
    rows[name] = {
        "PLV": PhaseLockingValue(window=1600).run(left, right)[-1].phase_locking_value,
        "correlation": CorrelationsMatrix(window=1600, number_of_inputs=2).run(signal)[-1].correlations_matrix[0, 1],
        "interdependence": NonlinearInterdependence(window=300, hop=100, embedding_dimension=3, neighbours=5).run(left, right).first_to_second_interdependence[-1],
        "cross-recurrence, diagonal": np.diag(CrossRecurrencePlot(window=500, epsilon=0.05).run(signal[:500, 0:1], signal[:500, 1:2])[-1].cross_recurrence_matrix).mean(),
    }
print(f"{'measure':<28}" + "".join(f"{name:>28}" for name in rows))
for measure in rows["trial 5 (synchronised)"]:
    print(f"{measure:<28}" + "".join(f"{float(rows[name][measure]):>28.2f}" for name in rows))

## Try it
1. Compute the PLV over the whole 16 s for trials 3 (synchronised) and 4 (unsynchronised), the "open the window" task. Which
   is higher?
2. In the cross-recurrence plot of trial 5, make ``epsilon`` 0.01 and 0.15 m. What happens to the diagonal, and why?
3. Ask ``GeneralisedAutoCorrelation`` for the recurrence rate at the single lag of 1.5 s (``output_type="scalar"``) in the two
   trials.



## Solutions
Exercise 1. The same call on another pair of trials.



In [ ]:
for number in (3, 4):
    trial = load_qualisys_trial(number, start=5, stop=21).to_units("m")
    plv = PhaseLockingValue(window=1600).run(trial.joint("Left_Hand")[:, 1], trial.joint("Right_Hand")[:, 1])[-1].phase_locking_value
    print(f"trial {number}: PLV over 16 s = {float(plv):.2f}")

Exercise 2. A small radius asks for an exact return, so fewer samples recur; a large radius counts almost every sample.



In [ ]:
synchronised = heights["trial 5 (synchronised)"]
for epsilon in (0.01, 0.05, 0.15):
    cross = CrossRecurrencePlot(window=500, epsilon=epsilon).run(synchronised[:500, 0:1], synchronised[:500, 1:2])[-1].cross_recurrence_matrix
    print(f"epsilon {epsilon} m: {cross.mean():.2f} of the plot recurs, {np.diag(cross).mean():.2f} of the main diagonal")

Exercise 3. A scalar output holds the rate at one lag, ``tau``, in samples.



In [ ]:
for name, position in clapping.items():
    scalar = GeneralisedAutoCorrelation(window=1000, epsilon=0.05, output_type="scalar", tau=77).run(position)[-1]
    print(f"{name}: recurrence rate at {int(scalar.taus[0]) / sampling_rate:.1f} s = {float(scalar.recurrence_rate[0]):.2f}")

## Next steps
* :doc:`/theory/synchronization`: definitions, choosing a measure and practical notes.
* :doc:`/api/analysis_primitives`: every parameter of these features.
* `sphx_glr_tutorials_05_analysis_primitives_plot_09b_synchronization_events.py`: synchronization of event trains, a
  sequence of events and several measures in a staged process.

